In [1]:
import echopype as ep

ed = ep.open_raw("./D20240904-T121523.raw", sonar_model="EK80")  # the tiny 15-record file is ideal

print(ed["Sonar"]["waveform_encode_descr"].values)   # e.g. ['complex_CW', 'power']

for g in sorted(p for p in ed.group_paths if "Beam_group" in p):
    ds = ed[g]
    kind = "complex" if "backscatter_i" in ds else "power"
    print(g, "->", kind, "| channels:", list(ds["channel"].values))


['power']
Sonar/Beam_group1 -> power | channels: [np.str_('WBT 400479-15 ES18_ES'), np.str_('WBT 400503-15 ES70-7C_ES'), np.str_('WBT 400509-15 ES200-7C_ES'), np.str_('WBT 400517-15 ES120-7C_ES'), np.str_('WBT 400528-15 ES38-7_ES')]


In [2]:
import echopype as ep
ed = ep.open_raw("D20240904-T121523.raw", sonar_model="EK80")

print(ed["Sonar"]["waveform_encode_descr"].values)          # expect: ['power']
bg = ed["Sonar/Beam_group1"]
print("beam group channels:", list(bg["channel"].values))    # expect 5
print("has complex?", "backscatter_i" in bg)                 # expect False


['power']
beam group channels: [np.str_('WBT 400479-15 ES18_ES'), np.str_('WBT 400503-15 ES70-7C_ES'), np.str_('WBT 400509-15 ES200-7C_ES'), np.str_('WBT 400517-15 ES120-7C_ES'), np.str_('WBT 400528-15 ES38-7_ES')]
has complex? False


In [4]:
import yaml, traceback
import echopype as ep
from aa_si_calibration.calibration import extract_standardized_calibration_parameters

RAW = r"D20240904-T121523.raw"   # <-- your local raw file (the tiny one is fine)
OUT = r"../outputs/calibration/mapping_files"   # <-- adjust to your outputs path

with open(f"{OUT}/calibration_configurations.yaml") as f:
    calibration_dict = yaml.safe_load(f)
with open(f"{OUT}/channel_mapping.yaml") as f:
    mapping_dict = yaml.safe_load(f)

ed = ep.open_raw(RAW, sonar_model="EK80")

res = extract_standardized_calibration_parameters(calibration_dict, mapping_dict, echodata=ed)
cal_params, env_params = res["cal_params"], res["env_params"]

print("beam group n_channels:", ed["Sonar/Beam_group1"]["channel"].size)
print("beam channel order   :", list(ed["Sonar/Beam_group1"]["channel"].values))
print("\n--- cal_params (this is what triggers param2da) ---")
for k, v in cal_params.items():
    L = len(v) if isinstance(v, (list, tuple)) else "scalar"
    print(f"{k:32s} len={L!s:>6}   {v}")
print("\n--- env_params ---")
for k, v in env_params.items():
    L = len(v) if isinstance(v, (list, tuple)) else "scalar"
    print(f"{k:32s} len={L!s:>6}   {v}")

print("\n--- compute_Sv full traceback (if it fails) ---")
try:
    ds = ep.calibrate.compute_Sv(ed, waveform_mode="CW", encode_mode="power",
                                 cal_params=cal_params, env_params=env_params)
    print("SUCCESS:", list(ds.data_vars))
except Exception:
    traceback.print_exc()


beam group n_channels: 5
beam channel order   : [np.str_('WBT 400479-15 ES18_ES'), np.str_('WBT 400503-15 ES70-7C_ES'), np.str_('WBT 400509-15 ES200-7C_ES'), np.str_('WBT 400517-15 ES120-7C_ES'), np.str_('WBT 400528-15 ES38-7_ES')]

--- cal_params (this is what triggers param2da) ---
gain_correction                  len=     5   [23.14, 27.54, 27.15, 27.05, 26.82]
sa_correction                    len=     5   [0.2092, -0.0532, -0.111, -0.1274, -0.1825]
equivalent_beam_angle            len=     5   [-17.0, -20.7, -20.7, -20.7, -20.7]
beamwidth_athwartship            len=     5   [11.01, 6.67, 6.44, 6.45, 6.84]
beamwidth_alongship              len=     5   [10.09, 6.41, 6.27, 6.35, 6.52]
angle_offset_athwartship         len=     5   [-0.16, -0.02, -0.1, -0.05, -0.15]
angle_offset_alongship           len=     5   [-0.36, 0.01, 0.02, 0.07, 0.24]
angle_sensitivity_athwartship    len=     5   [None, None, None, None, None]
angle_sensitivity_alongship      len=     5   [None, None, None, None

In [5]:
import echopype as ep

files = ["D20240904-T121523.raw", "D20240904-T162420.raw", "D20240904-T171938.raw"]
eds = []
for fn in files:
    ed = ep.open_raw(fn, sonar_model="EK80")
    eds.append(ed)
    descr = ed["Sonar"]["waveform_encode_descr"].values
    print(f"\n=== {fn} ===  waveform_encode_descr = {descr}")
    for g in sorted(p for p in ed.group_paths if "Beam_group" in p):
        ds = ed[g]
        kind = "complex" if "backscatter_i" in ds else "power"
        print(f"   {g}: {kind:8s} n_ch={ds['channel'].size}  {list(ds['channel'].values)}")

# Now the combined object the pipeline actually calibrates:
combined = ep.combine_echodata(eds)
print("\n=== COMBINED ===  waveform_encode_descr =", combined["Sonar"]["waveform_encode_descr"].values)
for g in sorted(p for p in combined.group_paths if "Beam_group" in p):
    ds = combined[g]
    kind = "complex" if "backscatter_i" in ds else "power"
    print(f"   {g}: {kind:8s} n_ch={ds['channel'].size}  {list(ds['channel'].values)}")



=== D20240904-T121523.raw ===  waveform_encode_descr = ['power']
   Sonar/Beam_group1: power    n_ch=5  [np.str_('WBT 400479-15 ES18_ES'), np.str_('WBT 400503-15 ES70-7C_ES'), np.str_('WBT 400509-15 ES200-7C_ES'), np.str_('WBT 400517-15 ES120-7C_ES'), np.str_('WBT 400528-15 ES38-7_ES')]

=== D20240904-T162420.raw ===  waveform_encode_descr = ['power']
   Sonar/Beam_group1: power    n_ch=5  [np.str_('WBT 400479-15 ES18_ES'), np.str_('WBT 400503-15 ES70-7C_ES'), np.str_('WBT 400509-15 ES200-7C_ES'), np.str_('WBT 400517-15 ES120-7C_ES'), np.str_('WBT 400528-15 ES38-7_ES')]

=== D20240904-T171938.raw ===  waveform_encode_descr = ['power']
   Sonar/Beam_group1: power    n_ch=5  [np.str_('WBT 400479-15 ES18_ES'), np.str_('WBT 400503-15 ES70-7C_ES'), np.str_('WBT 400509-15 ES200-7C_ES'), np.str_('WBT 400517-15 ES120-7C_ES'), np.str_('WBT 400528-15 ES38-7_ES')]

=== COMBINED ===  waveform_encode_descr = ['power']
   Sonar/Beam_group1: power    n_ch=5  [np.str_('WBT 400479-15 ES18_ES'), np.str

In [6]:
import numpy as np
print("single-file filter_time count:", eds[0]["Vendor_specific"].sizes.get("filter_time"))
print("combined   filter_time count:", combined["Vendor_specific"].sizes.get("filter_time"))

v = combined["Vendor_specific"]
for var in v.data_vars:
    if "filter_time" in v[var].dims:
        a = v[var]
        identical = all(a.isel(filter_time=i).equals(a.isel(filter_time=0))
                        for i in range(a.sizes["filter_time"]))
        print(f"{var}: identical across filter_times? {identical}")


single-file filter_time count: 1
combined   filter_time count: 3
WBT_coeffs_real: identical across filter_times? False
WBT_coeffs_imag: identical across filter_times? False
PC_coeffs_real: identical across filter_times? False
PC_coeffs_imag: identical across filter_times? False
WBT_deci_fac: identical across filter_times? False
PC_deci_fac: identical across filter_times? False
